# Coleta de Dados do Ibovespa

Este notebook realiza a coleta automatizada dos dados históricos do Ibovespa (IBOV) a partir dos arquivos oficiais IndexReport disponibilizados pela B3.

O período de análise compreende janeiro de 2021 a dezembro de 2025.

Os dados coletados serão utilizados posteriormente na comparação entre o mercado acionário brasileiro, criptomoedas e indicadores macroeconômicos.

## Fonte dos dados

A B3 disponibiliza arquivos diários do tipo `BVBG.087.01 - IndexReport`, contendo informações dos índices de mercado.

Neste projeto, os arquivos são obtidos automaticamente para os dias úteis do período analisado. A rotina realiza o download, identifica o arquivo XML correspondente e extrai somente os registros referentes ao Ibovespa (IBOV).

Datas sem arquivo de pregão disponível são desconsideradas.

In [0]:
import pandas as pd
import requests
import zipfile
import io
import time
import random
import xml.etree.ElementTree as ET

## Configuração da conexão com a B3

É utilizada uma sessão HTTP para reaproveitar a conexão entre as requisições. Também é definido um cabeçalho de navegador para tornar as chamadas mais estáveis durante a coleta.

In [0]:
sessao_b3 = requests.Session()

sessao_b3.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/152.0.0.0 Safari/537.36"
    )
})

## Funções de processamento

As funções a seguir são responsáveis por:

1. remover os namespaces presentes nas tags XML;
2. localizar o registro correspondente ao IBOV;
3. realizar o download do arquivo diário da B3;
4. tratar arquivos XML armazenados diretamente ou dentro de arquivos ZIP adicionais.

In [0]:
def remover_namespace(tag):
    return tag.split("}")[-1]

In [0]:
def extrair_ibov_xml(conteudo_xml, data):

    root = ET.fromstring(conteudo_xml)

    for elemento in root.iter():

        if remover_namespace(elemento.tag) != "IndxInf":
            continue

        dados = {}

        for filho in elemento.iter():
            campo = remover_namespace(filho.tag)

            if filho.text and filho.text.strip():
                dados[campo] = filho.text.strip()

        if dados.get("TckrSymb") == "IBOV":

            return {
                "data": pd.Timestamp(data),
                "ativo": "IBOV",
                "abertura": float(dados["OpngPric"]),
                "minima": float(dados["MinPric"]),
                "maxima": float(dados["MaxPric"]),
                "fechamento": float(dados["ClsgPric"]),
                "valor_indice": float(dados["IndxVal"]),
                "oscilacao": float(dados["OscnVal"])
            }

    return None

In [0]:
def baixar_e_extrair_ibov(data, tentativas=1):

    nome_arquivo = f"IR{data.strftime('%y%m%d')}.zip"

    url = (
        "https://www.b3.com.br/pesquisapregao/download"
        f"?filelist={nome_arquivo}"
    )

    for tentativa in range(1, tentativas + 1):

        try:
            response = sessao_b3.get(
                url,
                timeout=60
            )

            # ZIP vazio possui aproximadamente 22 bytes
            if response.status_code == 200 and len(response.content) > 22:

                with zipfile.ZipFile(
                    io.BytesIO(response.content),
                    "r"
                ) as zip_externo:

                    for nome in zip_externo.namelist():

                        # Caso o XML esteja diretamente no ZIP
                        if nome.lower().endswith(".xml"):

                            resultado = extrair_ibov_xml(
                                zip_externo.read(nome),
                                data
                            )

                            if resultado is not None:
                                return resultado

                        # Caso exista outro ZIP dentro do arquivo retornado
                        elif nome.lower().endswith(".zip"):

                            conteudo_zip_interno = zip_externo.read(nome)

                            with zipfile.ZipFile(
                                io.BytesIO(conteudo_zip_interno),
                                "r"
                            ) as zip_interno:

                                for nome_interno in zip_interno.namelist():

                                    if nome_interno.lower().endswith(".xml"):

                                        resultado = extrair_ibov_xml(
                                            zip_interno.read(nome_interno),
                                            data
                                        )

                                        if resultado is not None:
                                            return resultado

            if tentativa < tentativas:
                time.sleep(1.5 * tentativa)

        except Exception:
            if tentativa < tentativas:
                time.sleep(1.5 * tentativa)

    return None

## Coleta histórica do Ibovespa

A coleta é realizada em duas passagens dentro do mesmo processo.

Na primeira passagem, todos os dias úteis entre 2021 e 2025 são consultados uma vez. Em seguida, somente as datas sem resultado são verificadas novamente com novas tentativas.

Essa estratégia reduz o tempo de execução e ajuda a tratar respostas temporariamente indisponíveis do servidor da B3.

In [0]:
def coletar_ibovespa_periodo(inicio, fim):

    datas = pd.bdate_range(
        start=inicio,
        end=fim
    )

    registros = []
    pendentes = []

    print("=== COLETA DO IBOVESPA ===")
    print("Dias úteis a verificar:", len(datas))

    # Primeira passagem
    for i, data in enumerate(datas, start=1):

        try:
            registro = baixar_e_extrair_ibov(
                data,
                tentativas=1
            )

            if registro is not None:
                registros.append(registro)
            else:
                pendentes.append(data)

        except Exception:
            pendentes.append(data)

        time.sleep(0.25)

        if i % 100 == 0:
            print(
                f"Primeira passagem: {i}/{len(datas)} | "
                f"Registros: {len(registros)}"
            )

    print("\nPrimeira passagem concluída.")
    print("Registros encontrados:", len(registros))
    print("Datas para nova tentativa:", len(pendentes))

    # Segunda passagem somente nas datas pendentes
    recuperados = []
    sem_dados = []

    for i, data in enumerate(pendentes, start=1):

        try:
            registro = baixar_e_extrair_ibov(
                data,
                tentativas=4
            )

            if registro is not None:
                recuperados.append(registro)
            else:
                sem_dados.append(data)

        except Exception:
            sem_dados.append(data)

        time.sleep(random.uniform(0.15, 0.35))

        if i % 100 == 0:
            print(
                f"Recuperação: {i}/{len(pendentes)} | "
                f"Recuperados: {len(recuperados)}"
            )

    todos_registros = registros + recuperados

    df = pd.DataFrame(todos_registros)

    df = (
        df
        .drop_duplicates(subset=["data"])
        .sort_values("data")
        .reset_index(drop=True)
    )

    print("\nColeta concluída.")
    print("Total de registros:", len(df))
    print("Datas sem arquivo disponível:", len(sem_dados))

    return df, sem_dados

In [0]:
df_ibov, datas_sem_pregao = coletar_ibovespa_periodo(
    inicio="2021-01-01",
    fim="2025-12-31"
)

In [0]:
display(df_ibov.head(10))

## Validação da qualidade dos dados

Após a coleta, são verificadas a quantidade de registros, cobertura temporal, presença de valores nulos, registros duplicados e distribuição anual das observações.

In [0]:
print("=== VALIDAÇÃO FINAL IBOVESPA ===")

print("\nTotal de registros:")
print(len(df_ibov))

print("\nPeríodo:")
print(
    df_ibov["data"].min(),
    "até",
    df_ibov["data"].max()
)

print("\nValores nulos:")
print(
    df_ibov[
        [
            "data",
            "ativo",
            "abertura",
            "minima",
            "maxima",
            "fechamento",
            "valor_indice",
            "oscilacao"
        ]
    ].isnull().sum()
)

print("\nDuplicados:")
print(
    df_ibov.duplicated(
        subset=["data"]
    ).sum()
)

print("\nQuantidade por ano:")
print(
    df_ibov
    .assign(ano=df_ibov["data"].dt.year)
    .groupby("ano")
    .size()
)

print("\nDatas úteis sem arquivo de pregão:")
print(len(datas_sem_pregao))

## Resultado da coleta

Os dados históricos do Ibovespa foram obtidos diretamente a partir dos arquivos oficiais disponibilizados pela B3.

A rotina automatizada contempla o download, leitura dos arquivos compactados, interpretação do XML, identificação do IBOV e consolidação dos registros em estrutura tabular.

Datas úteis sem arquivo disponível são preservadas como ausência de pregão e não recebem preenchimento artificial.

Os dados consolidados serão posteriormente persistidos e tratados nas camadas Bronze, Silver e Gold do pipeline.